# 09 - PAN-12 Stage 2 Features

This notebook converts PAN-12 conversations into one conversation-level dataset for the second stage. It uses only the provided predator-author lists and existing XML fields for labels and features.

Conversation labels identify the presence of a listed predator author. They do not override the Stage 1 purpose: Stage 1 explains stage progression inside each conversation, while Stage 2 estimates conversation-level parental-review risk.

In [1]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

import joblib
import numpy as np
import pandas as pd
from scipy import sparse

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR = WORKSPACE / "models"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_XML = WORKSPACE / "pan12-sexual-predator-identification-training-corpus-2012-05-01.xml"
TEST_XML = WORKSPACE / "pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"
TRAIN_PREDATORS = WORKSPACE / "pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt"
TEST_PREDATORS = WORKSPACE / "pan12-sexual-predator-identification-groundtruth-problem1.txt"
STAGE1_MODEL = MODEL_DIR / "stage1_tfidf_logistic.joblib"

for required in [TRAIN_XML, TEST_XML, TRAIN_PREDATORS, TEST_PREDATORS, STAGE1_MODEL]:
    if not required.exists():
        raise FileNotFoundError(f"Required input not found: {required}")

## PAN-12 parsing and supplied labels

Conversation labels come from the supplied predator ID lists. If a predator ID appears in a conversation, that conversation is positive. The test split uses problem 1; the training split uses the corpus predator list.

In [2]:
DOCTYPE_PATTERN = re.compile(r"<!DOCTYPE[^>]*>", flags=re.IGNORECASE | re.DOTALL)
BARE_AMPERSAND_PATTERN = re.compile(r"&(?!#\d+;|#x[0-9A-Fa-f]+;|[A-Za-z_][A-Za-z0-9_.-]*;)")


def parse_xml(path: Path) -> ET.Element:
    raw = DOCTYPE_PATTERN.sub("", path.read_text(encoding="utf-8-sig"))
    raw = BARE_AMPERSAND_PATTERN.sub("&amp;", raw)
    return ET.fromstring(raw)


def load_id_file(path: Path) -> set[str]:
    return {
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip() and not line.strip().startswith("#")
    }


def load_pan12(path: Path, split_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    root = parse_xml(path)
    message_rows = []
    conversation_rows = []
    for conversation in root.findall(".//conversation"):
        conversation_id = conversation.attrib["id"]
        parts = []
        for message in conversation.findall("./message"):
            line = int(message.attrib["line"])
            author_node = message.find("./author")
            time_node = message.find("./time")
            text_node = message.find("./text")
            author = "" if author_node is None or author_node.text is None else author_node.text.strip()
            time_text = "" if time_node is None or time_node.text is None else time_node.text.strip()
            text = "" if text_node is None or text_node.text is None else text_node.text.strip()
            message_rows.append({
                "split": split_name,
                "conversation_id": conversation_id,
                "line": line,
                "author": author,
                "time_text": time_text,
                "text": text,
            })
            parts.append(text)
        conversation_rows.append({
            "split": split_name,
            "conversation_id": conversation_id,
            "conversation_text": " ".join(parts),
        })
    return pd.DataFrame(message_rows), pd.DataFrame(conversation_rows)


train_messages, train_conversations = load_pan12(TRAIN_XML, "train")
test_messages, test_conversations = load_pan12(TEST_XML, "test")

train_predators = load_id_file(TRAIN_PREDATORS)
test_predators = load_id_file(TEST_PREDATORS)

for frame, conversation_frame, predator_ids in (
    (train_messages, train_conversations, train_predators),
    (test_messages, test_conversations, test_predators),
):
    conversation_authors = frame.groupby("conversation_id")["author"].apply(lambda authors: set(authors.dropna())).to_dict()
    conversation_frame["label"] = conversation_frame["conversation_id"].map(
        lambda conversation_id: int(bool(conversation_authors.get(conversation_id, set()) & predator_ids))
    )

messages = pd.concat([train_messages, test_messages], ignore_index=True)
conversations = pd.concat([train_conversations, test_conversations], ignore_index=True)

print("Train conversations:", len(train_conversations), "positive:", int(train_conversations["label"].sum()))
print("Test conversations:", len(test_conversations), "positive:", int(test_conversations["label"].sum()))

Train conversations: 66927 positive: 2016
Test conversations: 155128 positive: 3737


## Stage 1 probabilities

The Stage 1 model receives the same previous/target/next context representation used during training. PAN-12 test predictions may already exist from notebook 08, but predictions are regenerated if missing. Processing is batched to limit memory use.

In [3]:
messages = messages.sort_values(["split", "conversation_id", "line"]).copy()
messages["previous_text"] = messages.groupby(["split", "conversation_id"], sort=False)["text"].shift(1).fillna("")
messages["next_text"] = messages.groupby(["split", "conversation_id"], sort=False)["text"].shift(-1).fillna("")
messages["model_text"] = (
    "previous: " + messages["previous_text"] + " target: " + messages["text"] + " next: " + messages["next_text"]
)

stage1_model = joblib.load(STAGE1_MODEL)
stage1_classes = [str(stage) for stage in stage1_model.classes_]
INFERENCE_BATCH_SIZE = 500
probability_batches = []
for batch_start in range(0, len(messages), INFERENCE_BATCH_SIZE):
    batch = messages["model_text"].iloc[batch_start:batch_start + INFERENCE_BATCH_SIZE]
    probability_batches.append(stage1_model.predict_proba(batch))
probabilities = np.vstack(probability_batches)

messages["predicted_stage"] = [stage1_classes[index] for index in probabilities.argmax(axis=1)]
for column_index, stage in enumerate(stage1_classes):
    messages[f"stage_{stage}_probability"] = probabilities[:, column_index]

## Conversation-level feature extraction

Behavioral features are assigned only when observable. PAN-12 time strings provide valid relative timing; no date or timezone is invented. Numeric ratios are guarded against zero denominators. Stage onsets are computed from positive probabilities rather than hard predictions.

In [4]:
def parse_time_minutes(time_text: str) -> float:
    match = re.fullmatch(r"\s*(\d{1,2}):(\d{2})\s*", str(time_text))
    if not match:
        return np.nan
    hour, minute = map(int, match.groups())
    if hour > 23 or minute > 59:
        return np.nan
    return hour * 60 + minute


messages["time_minutes"] = messages["time_text"].map(parse_time_minutes)
messages["message_length"] = messages["text"].fillna("").str.len()
messages["is_question"] = messages["text"].fillna("").str.contains(r"\?", regex=True)

def first_probability_onset(group: pd.DataFrame, column: str, threshold: float = 0.50) -> float:
    ordered = group.sort_values("line")
    positions = np.flatnonzero(ordered[column].to_numpy() >= threshold)
    return float((positions[0] + 1) / len(ordered)) if len(positions) else np.nan


def conversation_features(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values("line")
    number_of_messages = len(ordered)
    text_lengths = ordered["message_length"]
    author_counts = ordered["author"].value_counts()
    valid_times = ordered["time_minutes"].dropna()

    author_message_share = (author_counts.max() / number_of_messages) if number_of_messages else 0.0
    features = {
        "message_count": number_of_messages,
        "mean_message_length": text_lengths.mean(),
        "std_message_length": text_lengths.std(ddof=0) if number_of_messages else 0.0,
        "author_count": ordered["author"].nunique(),
        "dominant_author_share": author_message_share,
        "question_share": ordered["is_question"].mean(),
        "fraction_predicted_200": (ordered["predicted_stage"] == "200").mean(),
        "fraction_predicted_600": (ordered["predicted_stage"] == "600").mean(),
        "fraction_predicted_900": (ordered["predicted_stage"] == "900").mean(),
        "stage_600_threshold_onset": first_probability_onset(ordered, "stage_600_probability"),
        "stage_900_threshold_onset": first_probability_onset(ordered, "stage_900_probability"),
        "mean_stage_600_probability": ordered["stage_600_probability"].mean(),
        "mean_stage_900_probability": ordered["stage_900_probability"].mean(),
        "max_stage_600_probability": ordered["stage_600_probability"].max(),
        "max_stage_900_probability": ordered["stage_900_probability"].max(),
        "night_time_share": (ordered["time_minutes"].between(22 * 60, 24 * 60 - 1).sum() / len(valid_times)) if len(valid_times) else np.nan,
        "duration_minutes": valid_times.max() - valid_times.min() if len(valid_times) >= 2 else np.nan,
    }
    return pd.Series(features)


feature_rows = (
    messages.groupby(["split", "conversation_id"], sort=False, group_keys=False)
    .apply(conversation_features)
    .reset_index()
)
conversation_features_frame = conversations.merge(feature_rows, on=["split", "conversation_id"], how="left")
conversation_features_frame = conversation_features_frame.replace([np.inf, -np.inf], np.nan)

display(conversation_features_frame.head())
print("Conversation feature rows:", len(conversation_features_frame))
print("Positive labels:", int(conversation_features_frame["label"].sum()))

C:\Users\Kruti\AppData\Local\Temp\ipykernel_4616\2517497474.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(conversation_features)


,split,conversation_id,conversation_text,label,message_count,mean_message_length,std_message_length,author_count,dominant_author_share,question_share,...,fraction_predicted_600,fraction_predicted_900,stage_600_threshold_onset,stage_900_threshold_onset,mean_stage_600_probability,mean_stage_900_probability,max_stage_600_probability,max_stage_900_probability,night_time_share,duration_minutes
0,train,e621da5de598c9321a1d505ea95e6a2d,Hola. hi. whats up? not a ton. you? same. bei...,0,62.0,32.580645,28.169732,2.0,0.516129,0.258065,...,0.532258,0.080645,0.290323,NaN,0.434142,0.198953,0.844577,0.446954,0.0,18.0
1,train,1f1298186cac7c8e97ec901f30aa47f4,asuu lonte koe hi,0,3.0,5.000000,2.943920,1.0,1.000000,0.000000,...,0.000000,0.000000,NaN,NaN,0.238699,0.141719,0.341536,0.217515,0.0,0.0
2,train,1d0d6eb4815de5e2b27d0c396abf9dc7,hi asl m or f m,0,4.0,3.000000,1.870829,2.0,0.500000,0.000000,...,0.000000,0.000000,NaN,NaN,0.161640,0.095435,0.202345,0.135971,0.0,0.0
3,train,a7d90f72e1260762785b92a0c54fc4bb,happy is ayuppie word. a yuppie * yuppie hehe....,0,30.0,15.366667,19.465325,2.0,0.733333,0.366667,...,0.366667,0.033333,0.166667,NaN,0.344556,0.184474,0.621767,0.375096,0.0,12.0
4,train,380743a89b0163cef563c61c988ca1fd,ask me 5 questions and i will answer them trut...,0,3.0,33.666667,17.556259,2.0,0.666667,0.333333,...,0.666667,0.333333,1.000000,NaN,0.493849,0.364271,0.638388,0.482856,0.0,0.0


Conversation feature rows: 222055
Positive labels: 5753


## Source-text baseline and saved artifacts

The raw conversation text remains available as a separate text-only baseline feature. Missing numeric features are left as NaN so downstream notebooks can choose and document a transparent imputation policy.

In [5]:
messages.to_csv(PROCESSED_DIR / "pan12_messages_with_stage1.csv", index=False)
conversation_features_frame.to_csv(PROCESSED_DIR / "pan12_conversation_features.csv", index=False)

numeric_missingness = conversation_features_frame.isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame()
numeric_missingness.to_csv(REPORT_DIR / "pan12_conversation_feature_missingness.csv")

print("Saved PAN-12 Stage 1 message predictions and Stage 2 conversation features.")

Saved PAN-12 Stage 1 message predictions and Stage 2 conversation features.
